# Hotel Thread Analysis — Last Month Sample
**Data:** Feb 12 – Mar 11 2026 · 1 hotel · 765 bookings · 3281 guest messages  
**Pipeline:** Load → Threads → Classify (PROBLEM / QUESTION / OTHER) → Admin response time → Top question topics

In [2]:
!pip install xlrd


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# ── Cell 0: Load ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, json
from tqdm.auto import tqdm

PATH = "/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/Untitled.xls"

df = pd.read_excel(PATH, engine='xlrd')
df['date_add'] = pd.to_datetime(df['date_add'])
df['is_admin'] = df['from_admin'].fillna(0).astype(int)
df['message'] = df['message'].astype(str).str.strip()
df = df[df['message'].str.len() > 0].copy()

guest = df[df['is_admin'] == 0].copy()
admin = df[df['is_admin'] == 1].copy()

print(f"Date range: {df['date_add'].min().date()} → {df['date_add'].max().date()}")
print(f"Hotels: {sorted(df['hotel_id'].unique().tolist())}")
print(f"Bookings: {df['ID_booking'].nunique()}")
print(f"Guest msgs: {len(guest)} | Admin msgs: {len(admin)}")

Date range: 2026-02-12 → 2026-03-11
Hotels: [5]
Bookings: 765
Guest msgs: 3281 | Admin msgs: 2557


In [4]:
# ── Cell 1: Thread detection ─────────────────────────────────────────────────
# New thread = gap > GAP_HOURS between any two consecutive messages in a booking
# Run on full df so admin replies don't accidentally split threads

GAP_HOURS = 4

df_sorted = df.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)
df_sorted['prev_time'] = df_sorted.groupby('ID_booking')['date_add'].shift(1)
df_sorted['gap_hrs'] = (
    (df_sorted['date_add'] - df_sorted['prev_time']).dt.total_seconds() / 3600
)
df_sorted['new_thread'] = df_sorted['prev_time'].isna() | (df_sorted['gap_hrs'] > GAP_HOURS)
df_sorted['thread_id'] = df_sorted.groupby('ID_booking')['new_thread'].cumsum()

n_threads = df_sorted.groupby(['ID_booking', 'thread_id']).ngroups
n_bookings = df_sorted['ID_booking'].nunique()
print(f"Threads detected: {n_threads}")
print(f"Avg threads per booking: {n_threads / n_bookings:.1f}")

sizes = df_sorted.groupby(['ID_booking', 'thread_id']).size()
print(f"Single-message threads: {(sizes == 1).sum()}")
print(f"Multi-message threads:  {(sizes > 1).sum()}")

Threads detected: 1653
Avg threads per booking: 2.2
Single-message threads: 575
Multi-message threads:  1078


In [5]:
# ── Cell 2: Build threads DataFrame ─────────────────────────────────────────
# One row per (hotel_id, ID_booking, thread_id)
# text = all guest messages in that thread joined — this is what we classify

guest_sorted = df_sorted[df_sorted['is_admin'] == 0].copy()

threads = (
    guest_sorted
    .groupby(['hotel_id', 'ID_booking', 'thread_id'])
    .agg(
        text=('message', lambda msgs: '\n---\n'.join(msgs)),
        n_guest_msgs=('message', 'count'),
        thread_start=('date_add', 'min'),
        thread_end=('date_add', 'max'),
    )
    .reset_index()
)

print(f"Threads to classify: {len(threads)}")
threads.head(3)

Threads to classify: 1114


,hotel_id,ID_booking,thread_id,text,n_guest_msgs,thread_start,thread_end
0,5,322478,1,"Добрый день, как обычно из номера сверху регул...",2,2026-02-15 21:34:49,2026-02-15 21:36:37
1,5,322478,2,Добрый день. В моем номере постоянно сильный ш...,9,2026-02-16 13:06:41,2026-02-16 16:39:58
2,5,322478,3,"Добрый день, круто, жаль прочитал поздно) к со...",1,2026-02-17 09:32:56,2026-02-17 09:32:56


In [6]:
# ── Cell 3: OpenAI classifier (batched) ──────────────────────────────────────
# Batching 20 threads per call → ~20x cheaper and faster than one-by-one

from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

SYSTEM = """
You classify hotel guest conversation threads. Messages are in Russian.

PROBLEM: Physical action required — someone must come or do something.
  Includes: cleaning, maintenance, broken items, wifi voucher delivery,
  key issues, noise requiring staff. Also: "can I book a cleaner".
  Rule: if resolving it requires a person to physically act → PROBLEM.

QUESTION: A text reply fully resolves it. No physical action needed.
  Includes: checkout time, hotel policy, parking, directions, restaurant
  hours, wifi password (just the password, not a voucher delivery).

OTHER: Fits neither category. Explain briefly in 'reason'.
  Includes: pure greeting, payment dispute, cancellation, review threat.

Input: JSON array of objects with 'i' (index) and 'text' (guest messages).
Output: JSON object with key 'results' containing array of same length/order:
  [{"i": 0, "category": "PROBLEM"|"QUESTION"|"OTHER", "reason": "...", "confidence": 0.0-1.0}]
Output ONLY the JSON object, no markdown, no explanation.
""".strip()


def classify_batch(thread_texts: list) -> list:
    """Classify a batch of threads. Returns list of result dicts."""
    items = [{"i": i, "text": t[:600]} for i, t in enumerate(thread_texts)]  # cap length
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": json.dumps(items, ensure_ascii=False)},
            ],
        )
        raw = json.loads(resp.choices[0].message.content)
        results = raw.get('results', list(raw.values())[0] if raw else [])
        return sorted(results, key=lambda x: x.get('i', 0))
    except Exception as e:
        return [{"i": i, "category": "ERROR", "reason": str(e), "confidence": 0.0}
                for i in range(len(thread_texts))]


print(f"Model: {MODEL}")
print("classify_batch() ready")

Model: gpt-4o-mini
classify_batch() ready


In [7]:
# ── Cell 4: Run classification ────────────────────────────────────────────────
# With BATCH_SIZE=20 and ~N threads this should take a few minutes, not hours

BATCH_SIZE = 20
CHECKPOINT_FILE = "threads_sample_partial.parquet"

categories, reasons, confidences = [], [], []
texts = threads['text'].tolist()

for start in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[start : start + BATCH_SIZE]
    results = classify_batch(batch)
    categories.extend([r.get('category', 'ERROR') for r in results])
    reasons.extend([r.get('reason', '') for r in results])
    confidences.extend([r.get('confidence', 0.0) for r in results])

    # Checkpoint every 200 threads
    if len(categories) % 200 < BATCH_SIZE:
        partial = threads.iloc[:len(categories)].copy()
        partial['category'] = categories
        partial['reason'] = reasons
        partial.to_parquet(CHECKPOINT_FILE, index=False)

threads['category'] = categories
threads['reason'] = reasons
threads['confidence'] = confidences

threads.to_parquet('threads_sample_classified.parquet', index=False)

print('\n=== Results ===')
print(threads['category'].value_counts())
print()
print((threads['category'].value_counts(normalize=True) * 100).round(1))

  0%|          | 0/56 [00:00<?, ?it/s]


=== Results ===
category
PROBLEM     586
QUESTION    432
OTHER        96
Name: count, dtype: int64

category
PROBLEM     52.6
QUESTION    38.8
OTHER        8.6
Name: proportion, dtype: float64


In [8]:
# ── Cell 5: Per-hotel breakdown ───────────────────────────────────────────────

hotel_breakdown = (
    threads.groupby(['hotel_id', 'category'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all columns exist
for col in ['PROBLEM', 'QUESTION', 'OTHER', 'ERROR']:
    if col not in hotel_breakdown.columns:
        hotel_breakdown[col] = 0

hotel_breakdown['total_threads'] = (
    hotel_breakdown['PROBLEM'] + hotel_breakdown['QUESTION'] +
    hotel_breakdown['OTHER'] + hotel_breakdown['ERROR']
)

for col in ['PROBLEM', 'QUESTION', 'OTHER']:
    hotel_breakdown[f'{col}_pct'] = (
        hotel_breakdown[col] / hotel_breakdown['total_threads'] * 100
    ).round(1)

print('=== Per-Hotel Category Breakdown ===')
print(hotel_breakdown[[
    'hotel_id', 'total_threads',
    'PROBLEM', 'PROBLEM_pct',
    'QUESTION', 'QUESTION_pct',
    'OTHER', 'OTHER_pct'
]].to_string(index=False))

=== Per-Hotel Category Breakdown ===
 hotel_id  total_threads  PROBLEM  PROBLEM_pct  QUESTION  QUESTION_pct  OTHER  OTHER_pct
        5           1114      586         52.6       432          38.8     96        8.6


In [9]:
# ── Cell 6: Admin response time ───────────────────────────────────────────────
# Pure timestamp math — no invented scoring

admin_sorted = df_sorted[df_sorted['is_admin'] == 1].copy()

first_guest = (
    guest_sorted.groupby(['hotel_id', 'ID_booking'])['date_add']
    .min().reset_index()
    .rename(columns={'date_add': 'first_guest_time'})
)
first_admin = (
    admin_sorted.groupby(['hotel_id', 'ID_booking'])['date_add']
    .min().reset_index()
    .rename(columns={'date_add': 'first_admin_time'})
)
admin_counts = (
    admin_sorted.groupby(['hotel_id', 'ID_booking'])
    .size().reset_index(name='n_admin_msgs')
)
guest_counts = (
    guest_sorted.groupby(['hotel_id', 'ID_booking'])
    .size().reset_index(name='n_guest_msgs')
)

booking_stats = (
    first_guest
    .merge(first_admin, on=['hotel_id', 'ID_booking'], how='left')
    .merge(admin_counts, on=['hotel_id', 'ID_booking'], how='left')
    .merge(guest_counts, on=['hotel_id', 'ID_booking'], how='left')
)

booking_stats['n_admin_msgs'] = booking_stats['n_admin_msgs'].fillna(0).astype(int)
booking_stats['admin_responded'] = booking_stats['n_admin_msgs'] > 0
booking_stats['reply_time_min'] = (
    (booking_stats['first_admin_time'] - booking_stats['first_guest_time'])
    .dt.total_seconds() / 60
).round(1)
booking_stats.loc[booking_stats['reply_time_min'] < 0, 'reply_time_min'] = np.nan

total = len(booking_stats)
responded = booking_stats['admin_responded'].sum()
print('=== Admin Response Stats ===')
print(f"Bookings with reply: {responded} / {total} ({responded/total*100:.1f}%)")
print(f"Mean reply time:     {booking_stats['reply_time_min'].mean():.0f} min")
print(f"Median reply time:   {booking_stats['reply_time_min'].median():.0f} min")

print('\n=== Per Hotel ===')
hotel_stats = booking_stats.groupby('hotel_id').agg(
    total_bookings=('ID_booking', 'count'),
    response_rate=('admin_responded', 'mean'),
    mean_reply_min=('reply_time_min', 'mean'),
    median_reply_min=('reply_time_min', 'median'),
).reset_index()
hotel_stats['response_rate'] = (hotel_stats['response_rate'] * 100).round(1)
hotel_stats = hotel_stats.round(1)
print(hotel_stats.to_string(index=False))

=== Admin Response Stats ===
Bookings with reply: 425 / 433 (98.2%)
Mean reply time:     432 min
Median reply time:   4 min

=== Per Hotel ===
 hotel_id  total_bookings  response_rate  mean_reply_min  median_reply_min
        5             433           98.2           431.8               4.4


In [10]:
# ── Cell 7: Top question topics (TF-IDF) ─────────────────────────────────────
# No API cost — statistics only

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

question_texts = threads[threads['category'] == 'QUESTION']['text'].tolist()
print(f"QUESTION threads: {len(question_texts)}")

# Adjust N_TOPICS based on volume — don't cluster fewer points than clusters
N_TOPICS = min(20, max(5, len(question_texts) // 10))
print(f"Clustering into {N_TOPICS} topics")

vec = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X = vec.fit_transform(question_texts)

km = KMeans(n_clusters=N_TOPICS, random_state=42, n_init=10)
km.fit(X)

terms = vec.get_feature_names_out()
print(f'\n=== Top {N_TOPICS} Question Topics ===')
for i, center in enumerate(km.cluster_centers_):
    top_idx = center.argsort()[-7:][::-1]
    top_terms = [terms[j] for j in top_idx]
    cluster_size = (km.labels_ == i).sum()
    print(f"Topic {i+1:2d} ({cluster_size:3d} threads): {', '.join(top_terms)}")

QUESTION threads: 432
Clustering into 20 topics

=== Top 20 Question Topics ===
Topic  1 ( 27 threads): большое, спасибо, спасибо большое, если, на, ли, что
Topic  2 ( 52 threads): здравствуйте, как, за, не, по, на, что
Topic  3 ( 24 threads): здравствуйте подскажите, подскажите, подскажите пожалуйста, пожалуйста, здравствуйте, зал, прачечной
Topic  4 (  8 threads): выезд, поздний выезд, поздний, продлением, последующим продлением, последующим, доброе
Topic  5 ( 19 threads): уборка, когда, по, подскажите, пожалуйста когда, графику, по графику
Topic  6 ( 21 threads): можете, нужно, здравствуйте можете, здравствуйте, на, скинуть, мне
Topic  7 ( 12 threads): здравствуйте можно, можно, здравствуйте, интернета, для, можно пароль, для интернета
Topic  8 ( 12 threads): после, апреля, до, сегодня, сегодня после, до апреля, до 12
Topic  9 ( 33 threads): номер, скажите пожалуйста, скажите, аренды, договор, договор аренды, продлить
Topic 10 ( 12 threads): пришлите, пришлите пожалуйста, wifi, пожа

In [11]:
# ── Cell 8: OTHER threads — inspect ──────────────────────────────────────────

other = threads[threads['category'] == 'OTHER'][['ID_booking', 'thread_id', 'text', 'reason', 'confidence']]
print(f"OTHER threads: {len(other)}")
print()
for _, row in other.head(20).iterrows():
    preview = row['text'][:120].replace('\n', ' ')
    print(f"[booking {row['ID_booking']} / thread {row['thread_id']}]")
    print(f"  Reason:  {row['reason']}")
    print(f"  Preview: {preview}")
    print()

OTHER threads: 96

[booking 457247 / thread 1]
  Reason:  Вопрос о возврате залога и осмотре номера не требует физического действия.
  Preview: Здравствуйте, подскажите , пожалуйста , а вы мне залог вернете при сдаче номера ? --- последний раз мы платили стоимость

[booking 459524 / thread 3]
  Reason:  Проблема с машинкой решена, не требует дальнейших действий.
  Preview: Добрый вечер, машинка тау и не открылась --- Всё получилось, спасибо

[booking 459912 / thread 3]
  Reason:  Вопрос о звонке не требует физического действия.
  Preview: Добрый вечер, подскажите, только заметили, а по какому вопросу сегодня звонили с вертикаля в 12:00?

[booking 459912 / thread 10]
  Reason:  Запросы о бронировании и ценах не требуют физического действия.
  Preview: Доброго дня, подскажите, а бронь на лето в долгосрок в 1к ещё не открылось? --- Можете узнать на стандарты с раздельными

[booking 459912 / thread 15]
  Reason:  Запрос на помощь с пакетами не требует физического действия.
  Preview: Да, к